## 0. Imports & setup

We import `sentence-transformers` for MiniLM encoding and `numpy` for all embedding arithmetic.
`DATA_DIR` points to the same folder used in `1_prepare_data.ipynb` so all `.npy` files land next to `movies_clean.csv`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

DATA_DIR = Path('../data')

## 1. Load cleaned movie data

Load `movies_clean.csv` produced by `1_prepare_data.ipynb`. The file contains:
- **title, genres** — from MovieLens
- **overview, tagline, release_year, runtime, vote_average, vote_count, popularity, budget, revenue** — from TMDB
- **director, writer, composer, cinematographer, producer, cast** — from TMDB credits
- **keywords** — thematic tags from TMDB
- **country, origin_country, spoken_languages** — geographic info from TMDB
- **collection, production_companies** — franchise and studio info
- **poster_path, backdrop_path** — image paths used by the Telegram bot
- **text** — combined string `title | genres | overview` used as input to the sentence encoder

In [ ]:
movies = pd.read_csv(DATA_DIR / 'movies_clean.csv')
print(movies.shape)
movies[['title', 'genres', 'release_year', 'director']].head()

## 2. Description embeddings — MiniLM (384-dim)

`all-MiniLM-L6-v2` is a lightweight BERT-based model fine-tuned for semantic similarity.
It encodes the `text` field (title + genres + overview) into a 384-dimensional vector where semantically similar texts land close together.

After encoding we **L2-normalise** each vector so that dot product = cosine similarity — required for correct weighted score combination later.

Result saved to `emb_description.npy`.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
texts = movies['text'].fillna('').tolist()

emb_desc = model.encode(texts, show_progress_bar=True, batch_size=64).astype(np.float32)
emb_desc /= np.linalg.norm(emb_desc, axis=1, keepdims=True)

np.save(DATA_DIR / 'emb_description.npy', emb_desc)
print('Description embeddings:', emb_desc.shape)

## 3. Genre embeddings — multi-hot (~20-dim)

Each movie is represented as a binary vector over all genres: `1` if the movie belongs to that genre, `0` otherwise.
For example *Inception* → `[0, 0, 1, 0, 1, ...]` (Action=0, Comedy=0, Sci-Fi=1, Thriller=1, ...).

L2-normalised so that cosine similarity correctly measures genre overlap between two movies.

Result saved to `emb_genre.npy`. Genre vocabulary saved to `genre_vocab.npy`.

In [ ]:
all_genres = sorted(set(
    g for genres in movies['genres'].dropna()
    for g in genres.split('|')
))
genre_to_idx = {g: i for i, g in enumerate(all_genres)}
print('Genres:', all_genres)

emb_genre = np.zeros((len(movies), len(all_genres)), dtype=np.float32)
for i, genres_str in enumerate(movies['genres'].fillna('')):
    for g in genres_str.split('|'):
        if g in genre_to_idx:
            emb_genre[i, genre_to_idx[g]] = 1.0

norms = np.linalg.norm(emb_genre, axis=1, keepdims=True)
norms[norms == 0] = 1
emb_genre /= norms

np.save(DATA_DIR / 'emb_genre.npy', emb_genre)
np.save(DATA_DIR / 'genre_vocab.npy', np.array(all_genres))
print('Genre embeddings:', emb_genre.shape)

## 4. Year — stored as actual years for Gaussian similarity

Unlike the other embeddings, year is kept as a raw scalar (e.g. 2010) rather than a normalised vector.
Similarity between two movies is computed as a **Gaussian kernel**: `exp(-(year_A - year_B)² / 2σ²)`.

With `σ=10` years: a movie from the same year scores 1.0, ±10 years scores ~0.6, ±20 years scores ~0.14.
This gives a smooth decay rather than a hard cutoff.

Result saved to `emb_year.npy`.

In [ ]:
years = pd.to_numeric(movies['release_year'], errors='coerce')
years = years.fillna(years.median()).values.astype(np.float32)

np.save(DATA_DIR / 'emb_year.npy', years)
print(f'Year range: {int(years.min())} – {int(years.max())},  median: {int(np.median(years))}')


## 5. Director embeddings — genre profile (same dim as genre)

We cannot embed a director's *name* meaningfully — 'Christopher Nolan' as a string carries no information about his style.
Instead, for each director we compute the **average genre vector** across all their films in the dataset.

This captures stylistic patterns: Nolan's profile will be dominated by Sci-Fi/Thriller, Spielberg's by Adventure/Drama, etc.
Two directors with similar genre profiles will score high on director similarity.

Result saved to `emb_director.npy`.

In [ ]:
# For each director: average genre vector across all their movies
director_profile = {}
for director, group in movies.groupby('director'):
    idxs = group.index.tolist()
    director_profile[director] = emb_genre[idxs].mean(axis=0)

emb_director = np.zeros_like(emb_genre)
for i, director in enumerate(movies['director']):
    if pd.notna(director) and director in director_profile:
        emb_director[i] = director_profile[director]

norms = np.linalg.norm(emb_director, axis=1, keepdims=True)
norms[norms == 0] = 1
emb_director /= norms

np.save(DATA_DIR / 'emb_director.npy', emb_director)
print('Director embeddings:', emb_director.shape)
print(f'Unique directors: {movies["director"].nunique()}')

## 6. Keywords embeddings — multi-hot (top 500)

TMDB keywords are thematic tags like `dreams`, `heist`, `memory`. There are tens of thousands of unique keywords across all movies, so we limit to the **top 500 most frequent** to keep the vector manageable.
A movie gets `1` for each keyword it has, `0` otherwise, then L2-normalised.

This is the richest signal for content similarity — genres say 'Sci-Fi', keywords say 'artificial intelligence|time loop|dystopia'.

Result saved to `emb_keywords.npy` and `keywords_vocab.npy`.

In [ ]:
from collections import Counter

# Count keyword frequency across all movies
all_kw = [
    kw
    for kws in movies['keywords'].dropna()
    for kw in kws.split('|')
]
top_keywords = [kw for kw, _ in Counter(all_kw).most_common(500)]
kw_to_idx = {kw: i for i, kw in enumerate(top_keywords)}
print(f'Top keywords (sample): {top_keywords[:20]}')

emb_keywords = np.zeros((len(movies), len(top_keywords)), dtype=np.float32)
for i, kws in enumerate(movies['keywords'].fillna('')):
    for kw in kws.split('|'):
        if kw in kw_to_idx:
            emb_keywords[i, kw_to_idx[kw]] = 1.0

norms = np.linalg.norm(emb_keywords, axis=1, keepdims=True)
norms[norms == 0] = 1
emb_keywords /= norms

np.save(DATA_DIR / 'emb_keywords.npy', emb_keywords)
np.save(DATA_DIR / 'keywords_vocab.npy', np.array(top_keywords))
print(f'Keywords embeddings: {emb_keywords.shape}')


## 7. Cast embeddings — genre profile (same approach as director)

We can't embed actor names directly. Instead, for each actor we compute the **average genre vector** of all their films — capturing their typical genre territory.
For each movie, we average the genre profiles of its top cast members.

Result saved to `emb_cast.npy`.

In [ ]:
# Build genre profile per actor across all their movies
actor_profile = {}
for _, row in movies.iterrows():
    if pd.isna(row.get('cast')):
        continue
    for actor in str(row['cast']).split('|'):
        actor = actor.strip()
        if actor not in actor_profile:
            actor_profile[actor] = []
        actor_profile[actor].append(emb_genre[row.name])

actor_profile = {a: np.mean(vecs, axis=0) for a, vecs in actor_profile.items()}

# Each movie's cast embedding = average of its actors' genre profiles
emb_cast = np.zeros_like(emb_genre)
for i, row in movies.iterrows():
    if pd.isna(row.get('cast')):
        continue
    actors = [a.strip() for a in str(row['cast']).split('|') if a.strip() in actor_profile]
    if actors:
        emb_cast[i] = np.mean([actor_profile[a] for a in actors], axis=0)

norms = np.linalg.norm(emb_cast, axis=1, keepdims=True)
norms[norms == 0] = 1
emb_cast /= norms

np.save(DATA_DIR / 'emb_cast.npy', emb_cast)
print(f'Cast embeddings: {emb_cast.shape}')
print(f'Unique actors: {len(actor_profile)}')


## 8. Origin country embeddings — multi-hot

Multi-hot vector over all unique country ISO codes (US, GB, FR, ...).
Movies with the same origin country will score high on this signal.

Result saved to `emb_country.npy`.

In [ ]:
all_countries = sorted(set(
    c.strip()
    for countries in movies['origin_country'].dropna()
    for c in str(countries).split('|')
))
country_to_idx = {c: i for i, c in enumerate(all_countries)}
print(f'Unique countries: {len(all_countries)}')

emb_country = np.zeros((len(movies), len(all_countries)), dtype=np.float32)
for i, countries in enumerate(movies['origin_country'].fillna('')):
    for c in str(countries).split('|'):
        c = c.strip()
        if c in country_to_idx:
            emb_country[i, country_to_idx[c]] = 1.0

norms = np.linalg.norm(emb_country, axis=1, keepdims=True)
norms[norms == 0] = 1
emb_country /= norms

np.save(DATA_DIR / 'emb_country.npy', emb_country)
print(f'Country embeddings: {emb_country.shape}')


## 9. Numeric embeddings — budget & popularity (Gaussian similarity)

Like year, budget and popularity are stored as raw scalars. Similarity is computed at search time using a Gaussian kernel.

- **Budget**: log-transformed before saving because it spans many orders of magnitude ($1M → $300M+). Missing values filled with the median.
- **Popularity**: TMDB's internal score. Also filled with median for missing values.

Results saved to `emb_budget.npy` and `emb_popularity.npy`.

In [ ]:
# Budget — log scale to compress the wide range
budget_raw = pd.to_numeric(movies['budget'], errors='coerce')
budget_log = np.log1p(budget_raw)  # log1p handles None→NaN gracefully
emb_budget = budget_log.fillna(budget_log.median()).values.astype(np.float32)
np.save(DATA_DIR / 'emb_budget.npy', emb_budget)
print(f'Budget: {budget_raw.notna().sum()} non-null values')

# Popularity — raw score
pop = pd.to_numeric(movies['popularity'], errors='coerce')
emb_popularity = pop.fillna(pop.median()).values.astype(np.float32)
np.save(DATA_DIR / 'emb_popularity.npy', emb_popularity)
print(f'Popularity: {pop.notna().sum()} non-null values')


## 10. Load saved embeddings

Run this cell to restore all embeddings from disk without recomputing them.
Useful when restarting the kernel — sections 2–9 can take several minutes to recompute.

In [ ]:
# Run this cell to reload all embeddings from disk without recomputing
emb_desc     = np.load(DATA_DIR / 'emb_description.npy')
emb_genre    = np.load(DATA_DIR / 'emb_genre.npy')
years        = np.load(DATA_DIR / 'emb_year.npy')
emb_director = np.load(DATA_DIR / 'emb_director.npy')
emb_keywords = np.load(DATA_DIR / 'emb_keywords.npy')
emb_cast     = np.load(DATA_DIR / 'emb_cast.npy')
emb_country  = np.load(DATA_DIR / 'emb_country.npy')
emb_budget   = np.load(DATA_DIR / 'emb_budget.npy')
emb_popularity = np.load(DATA_DIR / 'emb_popularity.npy')
print('Loaded all embeddings')


## 11. Weighted similarity search

`find_similar_weighted` combines all 9 embedding spaces into a single score using user-defined weights:

```
score = w_desc       × cosine(desc_A,     desc_B)        # MiniLM text
      + w_genre      × cosine(genre_A,    genre_B)       # multi-hot genres
      + w_year       × gaussian(year_A,   year_B)        # release year
      + w_director   × cosine(dir_A,      dir_B)         # director genre profile
      + w_keywords   × cosine(kw_A,       kw_B)          # thematic keywords
      + w_cast       × cosine(cast_A,     cast_B)        # cast genre profile
      + w_country    × cosine(country_A,  country_B)     # origin country
      + w_budget     × gaussian(budget_A, budget_B)      # log budget
      + w_popularity × gaussian(pop_A,    pop_B)         # TMDB popularity
```

No FAISS needed at this scale — a numpy dot product over 10k vectors is fast enough (<1ms).
Weights do **not** need to sum to 1, but it helps keep scores interpretable.

In [ ]:
def find_similar_weighted(
    movie_title,
    w_desc=0.4, w_genre=0.2, w_year=0.1, w_director=0.1,
    w_keywords=0.1, w_cast=0.05, w_country=0.025, w_budget=0.0, w_popularity=0.0,
    year_sigma=10, budget_sigma=5.0, popularity_sigma=20.0,
    k=10
):
    matches = movies[movies['title'].str.lower().str.contains(movie_title.lower())]
    if matches.empty:
        print(f'Not found: {movie_title}')
        return

    idx = matches.index[0]
    row = movies.iloc[idx]
    print(f'Query : {row["title"]}  ({str(row.get("release_year", ""))[:4]})  dir: {row.get("director", "")}')
    print(f'Weights: desc={w_desc} genre={w_genre} year={w_year} dir={w_director} kw={w_keywords} cast={w_cast} country={w_country} budget={w_budget} pop={w_popularity}\n')

    # Gaussian similarity for scalar features
    year_sim       = np.exp(-((years        - years[idx])        ** 2) / (2 * year_sigma       ** 2))
    budget_sim     = np.exp(-((emb_budget   - emb_budget[idx])   ** 2) / (2 * budget_sigma     ** 2))
    pop_sim        = np.exp(-((emb_popularity - emb_popularity[idx]) ** 2) / (2 * popularity_sigma ** 2))

    scores = (
        w_desc       * (emb_desc     @ emb_desc[idx])     +
        w_genre      * (emb_genre    @ emb_genre[idx])    +
        w_year       * year_sim                           +
        w_director   * (emb_director @ emb_director[idx]) +
        w_keywords   * (emb_keywords @ emb_keywords[idx]) +
        w_cast       * (emb_cast     @ emb_cast[idx])     +
        w_country    * (emb_country  @ emb_country[idx])  +
        w_budget     * budget_sim                         +
        w_popularity * pop_sim
    )
    scores[idx] = -1

    top_k = np.argsort(scores)[::-1][:k]
    for i in top_k:
        r = movies.iloc[i]
        print(f'{r["title"]:45s}  {str(r.get("release_year",""))[:4]}  '
              f'{str(r.get("director","")):25s}  score={scores[i]:.3f}  {r["genres"]}')


## 12. Test — default weights

In [ ]:
# Default weights
find_similar_weighted('Inception', w_desc=0.4, w_genre=0.3, w_year=0.2, w_director=0.1)


## 13. Test — year matters most

In [ ]:
# Year matters most
find_similar_weighted('Inception', w_desc=0.2, w_genre=0.3, w_year=0.6, w_director=0.2)


## 14. 3D visualisation (PaCMAP)

PaCMAP reduces each embedding space to 3D while preserving both local clusters and global structure.
We loop over all spaces so you can compare how movies group differently depending on the signal:
- **Description**: clusters by theme and tone
- **Genre**: tight clusters by genre combination
- **Director**: clusters by directorial style (genre profile)
- **Keywords**: clusters by thematic tags
- **Cast**: clusters by typical actor genre territory
- **Country**: clusters by origin country

The plot is interactive: drag to rotate, scroll to zoom. Hover over any point to see title, genres, director, and release year.

In [ ]:
import pacmap
import plotly.express as px

movies['genre_first'] = movies['genres'].str.split('|').str[0]

embedding_spaces = {
    'Description (MiniLM)':      emb_desc,
    'Genre (multi-hot)':         emb_genre,
    'Director (genre profile)':  emb_director,
    'Keywords (multi-hot)':      emb_keywords,
    'Cast (genre profile)':      emb_cast,
    'Country (multi-hot)':       emb_country,
}

for name, emb in embedding_spaces.items():
    coords = pacmap.PaCMAP(n_components=3, random_state=42).fit_transform(emb)
    movies['x3'] = coords[:, 0]
    movies['y3'] = coords[:, 1]
    movies['z3'] = coords[:, 2]
    fig = px.scatter_3d(
        movies, x='x3', y='y3', z='z3',
        hover_name='title',
        hover_data={'genres': True, 'director': True, 'release_year': True, 'x3': False, 'y3': False, 'z3': False},
        color='genre_first',
        opacity=0.7,
        title=f'3D embedding space — {name}',
        width=1100, height=750,
    )
    fig.update_traces(marker=dict(size=3))
    fig.show()


In [ ]:
reducer_3d = pacmap.PaCMAP(n_components=3, random_state=42)
coords_3d  = reducer_3d.fit_transform(emb_genre)

movies['x3'] = coords_3d[:, 0]
movies['y3'] = coords_3d[:, 1]
movies['z3'] = coords_3d[:, 2]

fig = px.scatter_3d(
    movies, x='x3', y='y3', z='z3',
    hover_name='title',
    hover_data={'genres': True, 'director': True, 'release_year': True, 'x3': False, 'y3': False, 'z3': False},
    color='genre_first',
    opacity=0.7,
    title='Movie embedding space 3D (PaCMAP)',
    width=1100, height=750,
)
fig.update_traces(marker=dict(size=3))
fig.show()